# Document

langchain_core.documents.Document 是 LangChain 框架中最核心的基础数据结构。

你可以把它理解为 LangChain 系统中的“原子”单位。无论数据来源是 PDF、网页、Word 文档还是数据库，一旦被 LangChain 的加载器（Loader）读取，它们最终都会被转化为标准的 Document 对象。

1. Document 的核心结构
Document 对象非常简洁，它主要由两个部分组成：

page_content (字符串):

这是文档的实际文本内容。

大语言模型（LLM）直接阅读和处理的就是这部分内容。

metadata (字典):

这是关于文档的元数据（Context）。

它用于存储来源信息，例如：文件路径、页码、作者、发布日期、URL 等。

作用： 在检索（Retrieval）时，元数据对于过滤（Filtering）和引用来源（Citations）至关重要。

2. 代码示例：如何创建和使用
你需要从 langchain_core.documents 导入它。

示例 1：最基础的手动创建
这是最简单的用法，通常用于测试或处理内存中的文本。

In [1]:
from langchain_core.documents import Document

# 创建一个文档对象
doc = Document(
    page_content="LangChain 是一个用于构建 LLM 应用的框架。",
    metadata={
        "source": "manual_entry", 
        "author": "Gemini", 
        "date": "2023-10-01"
    }
)

# 访问内容
print(f"内容: {doc.page_content}")
# 输出: 内容: LangChain 是一个用于构建 LLM 应用的框架。

# 访问元数据
print(f"来源: {doc.metadata['source']}")
# 输出: 来源: manual_entry

内容: LangChain 是一个用于构建 LLM 应用的框架。
来源: manual_entry


示例 2：模拟真实场景（RAG 流程）
在 RAG（检索增强生成）流程中，Document 起到了承上启下的作用。

流程： 原始文件 -> Document Loader -> Document对象 -> VectorStore

假设我们正在处理一个公司政策的 PDF 文件：

In [2]:
# 假设这是从 PyPDFLoader 加载出来的结果
docs_from_pdf = [
    Document(
        page_content="员工每年享有 15 天带薪年假。",
        metadata={"source": "policy_handbook.pdf", "page": 12, "section": "Leave Policy"}
    ),
    Document(
        page_content="报销申请需要在每月 25 号前提交。",
        metadata={"source": "policy_handbook.pdf", "page": 45, "section": "Reimbursement"}
    )
]

# 模拟向量数据库检索
# 假设用户问：“怎么请假？”
# 检索器找到了第一个文档
retrieved_doc = docs_from_pdf[0]

# 发送给 LLM 的提示词可能如下所示：
prompt = f"""
根据以下信息回答用户问题：
信息来源：{retrieved_doc.metadata['source']} (第 {retrieved_doc.metadata['page']} 页)
内容：{retrieved_doc.page_content}
"""

print(prompt)


根据以下信息回答用户问题：
信息来源：policy_handbook.pdf (第 12 页)
内容：员工每年享有 15 天带薪年假。



3. Document 的关键作用作用领域具体描述
    - 标准化接口无论你用 TextLoader、PyPDFLoader 还是 WebBaseLoader，它们的输出永远是 List[Document]。这让后面的组件（如文本分割器）无需关心原始数据格式。
    - 文本分割 (Splitting)当使用 RecursiveCharacterTextSplitter 切分长文本时，它会保留原始文档的 metadata，确保切分后的每一个小片段都知道自己来自哪里。
    - 向量存储 (Vector Stores)当你把数据存入 ChromaDB 或 Pinecone 时，Embedding 向量对应的是 page_content，而 metadata 会被作为过滤条件存储（例如：只搜索 year: 2024 的文档）。

4. 进阶：由加载器生成的 Document
通常你不需要手动 new Document(...)，而是使用加载器。

In [3]:
from langchain_community.document_loaders import TextLoader

# 创建一个临时文件用于演示
with open("example.txt", "w", encoding="utf-8") as f:
    f.write("这是来自文件的具体内容。")

# 使用加载器
loader = TextLoader("example.txt", encoding="utf-8")
documents = loader.load()

print(documents)

[Document(metadata={'source': 'example.txt'}, page_content='这是来自文件的具体内容。')]
